# atlm_pro MP2 — Alignment: SFT then DPO

**Goal.** Take the MP1 continued-pretrained `SmolLM2-135M` — currently a
job-text *completer* — and align it into a **recruiter-query → job-description
generator**, in two stages:

1. **SFT** — supervised fine-tuning on `(recruiter query, job description)`
   pairs, teaching the model to *answer a request* rather than merely continue
   text.
2. **DPO** — preference alignment by RLAIF: the `atlm_teacher` (gemma-4) judges
   pairs of the SFT model's outputs, and DPO shifts the model toward the
   preferred one.

**Evaluation.** A three-way comparison — base `SmolLM2-135M` / MP1+SFT /
DPO-aligned — via gemma-4 LLM-as-judge win-rates and sample generations.

**Input data.** `data/processed/converted.jsonl` — 2,507 records, each a job
description paired with 3 synthetic recruiter queries (generated in the Stage-3
ETL by the gemma-4 teacher).

Part **MP2** of the three-part project: MP1 continued pretraining →
**MP2 alignment** → Final system.

## 0. Setup

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 120)

# Locate the project root (the folder that contains data/).
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA = PROJECT_ROOT / "data" / "processed"     # MP2 works from the processed corpus
OUTPUTS = PROJECT_ROOT / "outputs"
SRC = PROJECT_ROOT / "src"

print("Project root  :", PROJECT_ROOT)
print("Processed data:", DATA)
assert (DATA / "converted.jsonl").exists(), \
    "converted.jsonl not found — the Stage-3 query->JD ETL must be run first" 